# Exercise - Car engine condition classification using LSTM

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Load and preprocess data
Dataset: https://www.timeseriesclassification.com/description.php?Dataset=FordA

In [ ]:
def load_ucr_txt(path):
    """
    Args:
        path: filepath of the .txt file to load.
    """
    data = np.loadtxt(path)

    y = data[:, 0]
    X = data[:, 1:]

    return X, y

In [ ]:
# TODO: Load the dataset as X_train, y_train for the train and X_test, y_test for the test data.
# TODO: Inspect the data: 
#   How many train and test instances are contained in the dataset? How many positive and negative examples are contained in the dataset? 
#   How long are the sequences? What is the range of values? 
#   Plot examplary instances of both classes.

## Initialize PyTorch Datasets and Dataloaders

In [ ]:
# TODO: Add comments to the FordA dataset definition. Which transformations are applied to the data?

class FordADataset(Dataset):
    def __init__(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)

        mean = X.mean(dim=1, keepdim=True)
        std = X.std(dim=1, keepdim=True)
        X = (X - mean) / (std + 1e-8)

        X = X.unsqueeze(-1)

        y = ((torch.tensor(y) + 1) // 2).long()

        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_ds = FordADataset(X_train, y_train)
test_ds = FordADataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

## Define the LSTM classifier

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        """
        Args
            input_size: number of features per timestep
            hidden_size: hidden dimensionality of the LSTM
            num_classes: Number of classes in the dataset
        """
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, time, features)
        out, _ = self.lstm(x)  # Apply LSTM, out: (batch, time, hidden)
        pooled = out.mean(dim=1)  # Average over the timesteps
        pooled = self.norm(pooled)  # Apply layer norm to stabilize training 
        pooled = self.dropout(pooled)  # Apply dropout for regularization
        logits = self.fc(pooled)  # Final linear layer, outputs one score for each class
        return logits

In [ ]:
# TODO: Initialize the model. Use a hidden dimensionality of 64 for the LSTM.

## Train the model

In [ ]:
# TODO: Add comments to the training function. In particular, mark the steps that are listed in the pseudo code in the lecture slides.

def train(model, train_loader, num_epochs, device='cpu'):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0.0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(X)

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

    return model

In [ ]:
# TODO: Train the model for 50 epochs. For faster training, you may use a GPU, if available. In case of memory errors, try reducing the batch size or hidden dimensionality.

## Evaluate the model

In [ ]:
# TODO: Evaluate the trained model on the train and test sets in terms of accuracy.